# Interaktivna sveska — Chambolle TV, Gaus i median

Prikaz: Original | Šum | Gaus | Median | Chambolle.

Kopija gaussian_median_chambolle.ipynb, ali parametre menjaš slajderima (primeni se kad pustiš miš).

Potrebno: python -m pip install ipywidgets


In [4]:

import os
import io
import matplotlib
matplotlib.use('Agg')
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from IPython.display import display, clear_output

from skimage import data
from skimage.filters import gaussian, median
from skimage.morphology import footprint_rectangle

from ipywidgets import Dropdown, IntSlider, FloatSlider, VBox, Image as WImage




In [5]:
IMAGE_PATHS = ['flowers.jpg', 'landscapes.jpg', 'profile.jpg']

def load_image(image_path):
    if os.path.exists(image_path):
        image = Image.open(image_path).convert('L')
        return np.asarray(image, dtype=np.float64)
    return np.asarray(data.camera(), dtype=np.float64)

def add_gaussian_noise(image, std_dev, seed=0):
    rng = np.random.default_rng(seed)
    noisy_image = image + rng.normal(0.0, std_dev, image.shape)
    return np.clip(noisy_image, 0.0, 255.0)

def denoise_gaussian_filter(image, std_dev):
    return gaussian(image, sigma=std_dev, preserve_range=True)

def denoise_median_filter(image, size):
    return median(image, footprint=footprint_rectangle((size, size)))

def forward_gradient(u):
    gx = np.zeros_like(u)
    gy = np.zeros_like(u)
    gx[:, :-1] = u[:, 1:] - u[:, :-1]
    gy[:-1, :] = u[1:, :] - u[:-1, :]
    return gx, gy

def divergence(px, py):
    div = np.zeros_like(px)
    div[:, 0] = px[:, 0]
    div[:, 1:] = px[:, 1:] - px[:, :-1]
    div[0, :] += py[0, :]
    div[1:, :] += py[1:, :] - py[:-1, :]
    return div

def chambolle_projection(f, lam, dt=0.248, tol=1e-2, max_iter=2000, p=None):
    px = np.zeros_like(f) if p is None else p[0].copy()
    py = np.zeros_like(f) if p is None else p[1].copy()

    for _ in range(max_iter):
        gx, gy = forward_gradient(divergence(px, py) - f / lam)
        norm = np.sqrt(gx * gx + gy * gy)
        denom = 1.0 + dt * norm
        px_new = (px + dt * gx) / denom
        py_new = (py + dt * gy) / denom
        change = max(np.max(np.abs(px_new - px)), np.max(np.abs(py_new - py)))
        px, py = px_new, py_new
        if change <= tol:
            break

    u = np.clip(f - lam * divergence(px, py), 0.0, 255.0)
    return u, (px, py)

def denoise_tv(noisy_image, std_dev=20, n_outer=5, lam=None):
    sigma = std_dev
    n = np.sqrt(noisy_image.size)
    if lam is None:
        lam = 1.0 / (2.1237 / std_dev + 2.0547 / (std_dev ** 2))
    p = None
    u = noisy_image
    for _ in range(n_outer):
        u, p = chambolle_projection(noisy_image, lam, p=p)
        residual = np.linalg.norm(u - noisy_image)
        lam = lam * (n * sigma) / residual
    return u


In [6]:
clear_output(wait=True)

_image_cache = {}
_noise_cache = {}

def original_of(path):
    if path not in _image_cache:
        _image_cache[path] = load_image(path)
    return _image_cache[path]

def noisy_of(path, std_dev):
    key = (path, int(std_dev))
    if key not in _noise_cache:
        seed = {'flowers.jpg': 1, 'landscapes.jpg': 2, 'profile.jpg': 3}.get(path, 0)
        _noise_cache[key] = add_gaussian_noise(original_of(path), std_dev, seed=seed)
    return _noise_cache[key]

preview = WImage(format='png')

slika_w = Dropdown(options=IMAGE_PATHS, value='flowers.jpg', description='Slika')
sigma_w = IntSlider(value=20, min=5, max=50, step=1, description='σ šuma',
                    continuous_update=False)
gaus_w = FloatSlider(value=0.5, min=0.2, max=3.0, step=0.1, description='Gaus σ',
                     continuous_update=False)
median_w = IntSlider(value=3, min=3, max=11, step=2, description='Median n',
                     continuous_update=False)
lam_w = IntSlider(value=20, min=5, max=50, step=1, description='TV λ',
                  continuous_update=False)

def render_preview():
    original = original_of(slika_w.value)
    noisy = noisy_of(slika_w.value, sigma_w.value)
    g = denoise_gaussian_filter(noisy, gaus_w.value)
    m = denoise_median_filter(noisy, median_w.value)
    u, _ = chambolle_projection(noisy, lam_w.value)

    images = [original, noisy, g, m, u]
    titles = [
        'Original',
        f'Šum σ={sigma_w.value}',
        f'Gaus σ={gaus_w.value:g}',
        f'Median {median_w.value}×{median_w.value}',
        f'Chambolle λ={lam_w.value}',
    ]

    n = len(images)
    h, w = original.shape
    cy, cx = h // 2, w // 2
    size = 100

    fig, axes = plt.subplots(2, n, figsize=(3.6 * n, 7))
    for i, (image, title) in enumerate(zip(images, titles)):
        axes[0, i].imshow(image, cmap='gray', vmin=0, vmax=255)
        axes[0, i].set_title(title)
        axes[0, i].axis('off')
        crop = image[max(0, cy - size):min(h, cy + size), max(0, cx - size):min(w, cx + size)]
        axes[1, i].imshow(crop, cmap='gray', vmin=0, vmax=255)
        axes[1, i].set_title('Uvećano (200×200)')
        axes[1, i].axis('off')
    fig.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight')
    plt.close(fig)
    preview.value = buf.getvalue()

def on_change(_change):
    render_preview()

display(VBox([slika_w, sigma_w, gaus_w, median_w, lam_w, preview]))
render_preview()
for w in (slika_w, sigma_w, gaus_w, median_w, lam_w):
    w.observe(on_change, names='value')
